In [ ]:
import random
import json
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
import matplotlib.pyplot as plt

import networkx as nx

try:
    from transformers import pipeline
except Exception:
    pipeline = None
from typing import List, Dict, Any, Optional
import networkx as nx
import json
import numpy
from google import genai
from dotenv import load_dotenv
import time
import networkx as nx

In [ ]:
load_dotenv()

### Workflow difficult
1. Scan graph with one agent, that then returns "suspicious" nodes/ areas/ connections
2. let repair graph repair those suspicious areas

### workflow easy
1. tell agent the rule and tell him to immediately repair the graph

## Load graph

In [ ]:
import json
import networkx as nx

def load_fz_json_graph(path: str) -> nx.MultiDiGraph:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    g = nx.MultiDiGraph()
    
    # 1. Add Nodes
    for node in data.get("nodes", []):
        node_id = node.get("id")
        # Extract all attributes except the id itself
        attrs = {k: v for k, v in node.items() if k != "id"}
        g.add_node(node_id, **attrs)
    
    # 2. Add Edges
    for edge in data.get("edges", []):
        src = edge.get("src")
        dst = edge.get("dst")
        relation = edge.get("relation")
        # Store the relation as an attribute on the edge
        g.add_edge(src, dst, relation=relation)
        
    return g

g_corrupted = load_fz_json_graph("corrupted_graphs/fz_corrupted_graph_20260127-111351.json")
print(f"Successfully loaded graph with {g_corrupted.number_of_nodes()} nodes.")
print(f"Successfully loaded graph with {g_corrupted.number_of_edges()} edges.")
print(type(g_corrupted))

In [ ]:
def get_restaurant_details(g: nx.MultiDiGraph, restaurant_node_id: str) -> dict:
    """
    Flattens the graph structure into a dictionary for the LLM.
    e.g., finds the 'city' node connected to fz_0 and gets its value.
    """
    details = dict(g.nodes[restaurant_node_id])
    
    # Look at outgoing edges to find attribute nodes (City, Addr, etc.)
    for _, neighbor_id, edge_data in g.out_edges(restaurant_node_id, data=True):
        rel = edge_data.get("relation")
        neighbor_data = g.nodes[neighbor_id]
        
        # If the neighbor has a 'value' (like the city name), add it to our dict
        if "value" in neighbor_data:
            details[rel] = neighbor_data["value"]
            
    return details

In [ ]:
details = get_restaurant_details(g_corrupted, "fz_0")
print(details)

## Repair Operations

In [ ]:
# reverse operations to to repair graph (reverse corruptions)
# schema drift: label change
@dataclass
class RepairOp:
    kind: str
    params: Dict[str, Any]

def schema_drift_repair(g: nx.MultiDiGraph, restaurant_id, to_label) -> nx.MultiDiGraph:
    for node_id, data in g.nodes(data=True):
        if data.get("entity_id") == restaurant_id:
            g.nodes[node_id]["label"] = to_label
    return g
    
# pipeline id collision: 
def pipeline_id_collision_repair(g: nx.MultiDiGraph, node_id: Any, new_unique_id: Any, attribute_to_split_by: str) -> nx.MultiDiGraph:

    return entity_merge_repair(g, node_id, new_unique_id, move_edges=[])

def entity_merge_repair(g: nx.MultiDiGraph, node_to_split: Any, new_node_id: Any, move_neighbor_ids: List[Any]) -> nx.MultiDiGraph:
    if node_to_split not in g: return g
    g.add_node(new_node_id, **g.nodes[node_to_split])
    
    # Identify edges connected to the neighbors the LLM pointed out
    for u, v, key, data in list(g.edges(node_to_split, keys=True, data=True)):
        if v in move_neighbor_ids or u in move_neighbor_ids:
            new_u = new_node_id if u == node_to_split else u
            new_v = new_node_id if v == node_to_split else v
            g.add_edge(new_u, new_v, key=key, **data)
            g.remove_edge(u, v, key)
    return g

# entity split
def _merge_nodes(g: nx.MultiDiGraph, keep: Any, drop: Any) -> None:
    keep_data = g.nodes[keep]
    drop_data = g.nodes[drop]
    for k, v in drop_data.items():
        if k not in keep_data:
            keep_data[k] = v
    for u, _, key, data in list(g.in_edges(drop, keys=True, data=True)):
        g.add_edge(u, keep, key=key, **data)
    for _, v, key, data in list(g.out_edges(drop, keys=True, data=True)):
        g.add_edge(keep, v, key=key, **data)
    g.remove_node(drop)

def entity_split_repair(g: nx.MultiDiGraph, keep_node: Any, drop_node: Any) -> nx.MultiDiGraph:
    """
    Reverses a split by merging the duplicate node back into the original.
    Uses your existing _merge_nodes helper logic.
    """
    if keep_node in g and drop_node in g:
        _merge_nodes(g, keep_node, drop_node)
    return g
# adversarial neighbor rewire
def neighbor_rewire_repair(g: nx.MultiDiGraph, source_node: Any, wrong_target: Any, correct_target: Any, edge_key: Any) -> nx.MultiDiGraph:
    """
    Moves an edge from an incorrect neighbor back to the correct one.
    """
    if g.has_edge(source_node, wrong_target, key=edge_key):
        edge_data = g.get_edge_data(source_node, wrong_target, key=edge_key)
        g.add_edge(source_node, correct_target, key=edge_key, **edge_data)
        g.remove_edge(source_node, wrong_target, key=edge_key)
    return g



## Repair Agent

In [ ]:
import json
import re
import random
import time
from typing import List, Dict, Any
from google.api_core import exceptions

def _extract_json_from_string(text: str) -> Dict[str, Any]:
    """Extracts and parses JSON from a string that may contain Markdown blocks."""
    try:
        # 1. Try to find content between ```json and ```
        match = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
        if match:
            json_str = match.group(1)
        else:
            # 2. If no backticks, try to find the first '{' and last '}'
            start = text.find('{')
            end = text.rfind('}') + 1
            json_str = text[start:end]
            
        return json.loads(json_str)
    except Exception as e:
        print(f"Error parsing string to JSON: {e}\nRaw Text: {text}")
        return {}

def extract_corrupted_ids(corrupted_log: dict) -> list:
    ids = set()
    for op in corrupted_log.get("ops", []):
        params = op.get("params", {})
        # Check common keys used for node identifiers
        for key in ["node", "node_a", "node_b"]:
            if key in params:
                ids.add(params[key])
    return list(ids)

In [ ]:
class RepairAgent:
    def __init__(self, model_name: str = "gemma-3-1b-it"):
        # Initialize the Gemini Client
        self.client = genai.Client()
        self.model_name = model_name
        self.rules = [
            "Same Name + Same City must have Same Area Code.",
            "Labels must match relationship types (e.g. WORKS_AT connects Person to Company)."
        ]

    def _format_neighborhood(self, g: nx.MultiDiGraph, node_id: str) -> str:
        """Helper to create a JSON string of a node's local context."""
        node_data = g.nodes[node_id]
        edges = []
        for _, v, data in g.out_edges(node_id, data=True):
            edges.append({"target": v, "relation": data.get("relation"), "target_data": g.nodes[v]})
        
        return json.dumps({"node_id": node_id, "data": node_data, "out_edges": edges}, indent=2)

    def detect(self, g: nx.MultiDiGraph, audit_size: int) -> List[Dict[str, Any]]:
        # 1. Use your heuristic code to find suspects first (saves money/time)
        suspect_ids = self.detect_suspects(g)
        if not suspect_ids:
            all_nodes = list(g.nodes())
            suspect_ids = random.sample(all_nodes, min(len(all_nodes), audit_size))
            print(f"Heuristics clean. Running random audit on {len(suspect_ids)} nodes...")
        findings = []

        for node_id in suspect_ids:
            neighborhood = self._format_neighborhood(g, node_id)
            # prompt = f"""Identify if this subgraph is corrupt.
            # Rules: {self.rules}
            # Subgraph: {neighborhood}
            # Return ONLY JSON: {{"is_corrupt": true, "type": "drift|collision|merge|split|rewire", "node": "{node_id}"}}"""
            

            prompt = f"""AUDIT TASK: Analyze this subgraph for structural or semantic corruption.
            SUBGRAPH DATA:
            {neighborhood}
            INSTRUCTIONS:
            1. Determine if a corruption exists.
            2. If yes, classify it into EXACTLY ONE of these categories:
            - 'drift': Label mismatch (e.g., a Restaurant labeled as 'Addr') (possible labels are only Name, Addr, Restaurant, Phone, Type, City).
            - 'collision': Two distinct entities merged due to shared IDs.
            - 'merge': Two different entities incorrectly merged into one node.
            - 'split': One entity incorrectly split into two nodes (look for duplicates).
            - 'rewire': An edge points to a nonsensical neighbor.
            3. If the subgraph is correct and follows all rules, set 'is_corrupt' to false and 'type' to 'none'.
            RESPONSE FORMAT (Strict JSON):
            {{
                "is_corrupt": boolean,
                "type": "drift" | "collision" | "merge" | "split" | "rewire" | "none",
                "node": "{node_id}",
                "reason": "Explain your choice"
            }}
            """

          
            # Gemini Call
            response = self.client.models.generate_content(
                model=self.model_name, 
                contents=prompt
                # config={'response_mime_type': 'application/json'}
            )
            print(response.text)
            
            analysis = _extract_json_from_string(response.text)
            if analysis.get("is_corrupt"):
                findings.append(analysis)
        
        return findings
    

    def detect_full_graph(self, g: nx.MultiDiGraph, batch_size: int = 10) -> List[Dict[str, Any]]:
        all_nodes = list(g.nodes())
        findings = []
        
        print(f"Starting full audit of {len(all_nodes)} nodes in batches of {batch_size}...")

        for i in range(0, len(all_nodes), batch_size):
            batch = all_nodes[i : i + batch_size]
            
            # We build a single large context for the batch
            batch_context = []
            for node_id in batch:
                batch_context.append(self._format_neighborhood(g, node_id))
            
            prompt = f"""AUDIT TASK: Analyze the following list of graph neighborhoods for corruption.
            
            DATA BATCH:
            {json.dumps(batch_context, indent=2)}

            INSTRUCTIONS:
            1. Evaluate each neighborhood.
            2. Classify errors into: 'drift', 'collision', 'merge', 'split', 'rewire'.
            3. If no corruption exists for a node, omit it from the results.

            RESPONSE FORMAT:
            Return a JSON list of objects. Each object MUST look like this:
            [
                {{
                    "is_corrupt": true,
                    "type": "drift",
                    "node": "node_id",
                    "reason": "..."
                }}
            ]
            If the entire batch is clean, return an empty list [].
            """

            try:
                response = self.client.models.generate_content(
                    model=self.model_name, 
                    contents=prompt
                )
                
                batch_results = _extract_json_from_string(response.text)
                
                if isinstance(batch_results, list):
                    for analysis in batch_results:
                        if analysis.get("is_corrupt"):
                            findings.append(analysis)
                
                time.sleep(2) 
                
            except Exception as e:
                print(f"Error in batch starting at index {i}: {e}")

        return findings
    

    def detect_sampled_graph(self, g: nx.MultiDiGraph, batch_size: int = 5, sample_pct: float = 1.0) -> List[Dict[str, Any]]:
        # 1. Select the nodes to audit
        all_node_ids = list(g.nodes())
        sample_size = int(len(all_node_ids) * sample_pct)
        audited_nodes = random.sample(all_node_ids, sample_size)
        
        findings = []
        total_batches = (len(audited_nodes) + batch_size - 1) // batch_size
        
        print(f"--- Starting Sampled Audit ---")
        print(f"Sampling {sample_pct*100}% of the graph.")
        print(f"Targeting {len(audited_nodes)} nodes ({total_batches} total batches).")

        # 2. Process in Batches
        for i in range(0, len(audited_nodes), batch_size):
            batch = audited_nodes[i : i + batch_size]
            current_batch_num = (i // batch_size) + 1
            
            # Prepare context for the LLM
            batch_context = [self._format_neighborhood(g, nid) for nid in batch]
            
            prompt = f"""AUDIT TASK: Analyze these graph neighborhoods for corruption.
            DATA BATCH:
            {json.dumps(batch_context, indent=2)}
            RESPONSE FORMAT (Strict JSON List):
            [ {{"is_corrupt": true, "type": "drift|merge|split|rewire", "node": "id", "reason": "..."}} ]
            """

            # 3. retry loop
            success = False
            while not success:
                try:
                    print(f"[{current_batch_num}/{total_batches}] Processing nodes {i} to {i+len(batch)}...", end="\r")
                    
                    response = self.client.models.generate_content(
                        model=self.model_name, 
                        contents=prompt
                    )
                    print(response.text)
                    batch_results = _extract_json_from_string(response.text)
                    if isinstance(batch_results, list):
                        findings.extend([res for res in batch_results if res.get("is_corrupt")])
                    
                    success = True
                    time.sleep(3)
                    
                except exceptions.ResourceExhausted as e:
                    print(f"\n[Quota Paused] Tokens/Minute exceeded. Waiting 45s...")
                    time.sleep(45) 
                    continue 
                    
                except Exception as e:
                    print(f"\n[Error] Skipping batch at index {i}: {e}")
                    break 

        print(f"\n--- Audit Complete ---")
        print(f"Total Corruptions Found: {len(findings)}")
        return findings
    
    def detect_hybrid_graph(self, g: nx.MultiDiGraph, corrupted_log: dict, batch_size: int = 5, sample_pct: float = 0.1) -> List[Dict[str, Any]]:
        # 1. Get the "Must-Audit" nodes from the log
        mandatory_ids = extract_corrupted_ids(corrupted_log)
        mandatory_ids = [nid for nid in mandatory_ids if nid in g]
        
        # 2. Calculate how many more nodes we need to reach the sample percentage
        all_node_ids = list(g.nodes())
        total_target_size = int(len(all_node_ids) * sample_pct)
        
        # 3. Select random nodes, excluding the ones we already have
        remaining_pool = list(set(all_node_ids) - set(mandatory_ids))
        random_needed = max(0, total_target_size) - len(mandatory_ids)
        random_ids = random.sample(remaining_pool, min(len(remaining_pool), random_needed))
        
        # 4. Final list to audit
        audited_nodes = list(set(mandatory_ids + random_ids))
        
        print(f"--- Starting Hybrid Audit ---")
        print(f"Mandatory nodes (from log): {len(mandatory_ids)}")
        print(f"Randomly sampled nodes: {len(random_ids)}")
        print(f"Total nodes to audit: {len(audited_nodes)}")

        # 2. Process in Batches
        findings = []
        total_batches = (len(audited_nodes) + batch_size - 1) // batch_size
        for i in range(0, len(audited_nodes), batch_size):
            batch = audited_nodes[i : i + batch_size]
            current_batch_num = (i // batch_size) + 1
            
            # Prepare context for the LLM
            batch_context = [self._format_neighborhood(g, nid) for nid in batch]
            
            # prompt = f"""AUDIT TASK: Analyze these graph neighborhoods for corruption.
            # DATA BATCH:
            # {json.dumps(batch_context, indent=2)}
            # RESPONSE FORMAT (Strict JSON List):
            # [ {{"is_corrupt": true, "type": "drift|merge|split|rewire", "node": "id", "reason": "..."}} ]
            # """
            prompt = f"""AUDIT TASK: Analyze these graph neighborhoods for structural or semantic corruption.
            DATA BATCH::
            {json.dumps(batch_context, indent=2)}
            INSTRUCTIONS:
            1. Determine if a corruption exists.
            2. If yes, classify it into EXACTLY ONE of these categories:
            - 'drift': Label mismatch (e.g., a Restaurant labeled as 'Addr') (possible labels are only Name, Addr, Restaurant, Phone, Type, City).
            - 'collision': Two distinct entities merged due to shared IDs.
            - 'merge': Two different entities incorrectly merged into one node.
            - 'split': One entity incorrectly split into two nodes (look for duplicates).
            - 'rewire': An edge points to a nonsensical neighbor.
            3. If the subgraph is correct and follows all rules, set 'is_corrupt' to false and 'type' to 'none'.
            4. The "node" field in your JSON response MUST EXACTLY MATCH the "node_id" provided in the DATA BATCH.
            5. Do not use generic numbers. If the data says "fz_353", your response must say "fz_353".
            RESPONSE FORMAT (Strict JSON List):
            [{{
                "is_corrupt": boolean,
                "type": "drift" | "collision" | "merge" | "split" | "rewire" | "none",
                "node": "EXACT_NODE_ID_FROM_DATA",
                "reason": "Explain your choice"
            }}]
            """

            # 3. Self-healing retry loop
            success = False
            while not success:
                try:
                    # Progress update
                    print(f"[{current_batch_num}/{total_batches}] Processing nodes {i} to {i+len(batch)}...", end="\r")
                    
                    response = self.client.models.generate_content(
                        model=self.model_name, 
                        contents=prompt
                    )
                    print(response.text)
                    batch_results = _extract_json_from_string(response.text)
                    if isinstance(batch_results, list):
                        findings.extend([res for res in batch_results if res.get("is_corrupt")])
                    
                    success = True
                    time.sleep(3) # Maintain 30 RPM
                    
                except exceptions.ResourceExhausted as e:
                    # Handle TPM/RPM limits
                    print(f"\n[Quota Paused] Tokens/Minute exceeded. Waiting 45s...")
                    time.sleep(45) 
                    continue 
                    
                except Exception as e:
                    print(f"\n[Error] Skipping batch at index {i}: {e}")
                    break 

            print(f"\n--- Audit Complete ---")
            print(f"Total Corruptions Found: {len(findings)}")
            return findings
    

    def plan(self, g: nx.MultiDiGraph, findings: List[Dict[str, Any]]) -> List[RepairOp]:
        planned_ops = []
        for issue in findings:
            neighborhood = self._format_neighborhood(g, issue['node'])
            # prompt = f"Propose a RepairOp for {issue['type']} at {issue['node']}. Neighborhood: {self._format_neighborhood(g, issue['node'])}"
            # prompt = f"""Propose a RepairOp for a {issue['type']} error at node {issue['node']}.
            # Neighborhood Context:
            # {neighborhood}

            # Return ONLY a JSON object compatible with a RepairOp:
            # {{"kind": "schema_drift_repair|entity_split_repair|neighbor_rewire_repair", 
            #   "params": {{ "node_id": "...", "to_label": "...", "etc": "..." }} }}"""
            


            prompt = f"""REPAIR PLANNER: Generate a precise RepairOp for the {issue['type']} error at node {issue['node']}.

            CONTEXT:
            {neighborhood}

            REPAIR MENU (Select the matching KIND and provide the REQUIRED PARAMS):

            1. If type is 'drift' -> KIND: "schema_drift_repair" (possible labels are only Name, Addr, Restaurant, Phone, Type, City)
            PARAMS: {{"restaurant_id": "the_entity_id_attribute", "to_label": "correct_label_string"}}

            2. If type is 'collision' -> KIND: "pipeline_id_collision_repair"
            PARAMS: {{"node_id": "{issue['node']}", "new_unique_id": "new_id_string", "attribute_to_split_by": "attr_name"}}

            3. If type is 'merge' -> KIND: "entity_merge_repair"
            PARAMS: {{"node_to_split": "{issue['node']}", "new_node_id": "new_id_string", "move_neighbor_ids": ["id1", "id2"]}}

            4. If type is 'split' -> KIND: "entity_split_repair"
            PARAMS: {{"keep_node": "original_node_id", "drop_node": "duplicate_node_id"}}

            5. If type is 'rewire' -> KIND: "neighbor_rewire_repair"
            PARAMS: {{"source_node": "{issue['node']}", "wrong_target": "id", "correct_target": "id", "edge_key": 0}}

            RESPONSE FORMAT (Strict JSON):
            {{
                "kind": "function_name_from_menu",
                "params": {{ ... }}
            }}
            """



            response = self.client.models.generate_content(
                model=self.model_name, 
                contents=prompt
                # config={'response_mime_type': 'application/json'}
            )
            plan_json = _extract_json_from_string(response.text)
            planned_ops.append(RepairOp(kind=plan_json["kind"], params=plan_json["params"]))
            print(response.text)
        return planned_ops
    

    def detect_suspects(self, g: nx.MultiDiGraph) -> List[Any]:
        suspects = []
        for n, data in g.nodes(data=True):
            # 1. Check for Label-Attribute Mismatch (Semantic drift)
            if data.get("label") == "Restaurant" and "salary" in data:
                suspects.append(n)
                continue
                
            # 2. Check for Functional Dependency violations
            out_edges = g.out_edges(n, data=True)
            relations = [d.get("relation") for _, _, d in out_edges]
            if len(relations) != len(set(relations)):
                # This node has duplicate relations (e.g., two 'city' edges)
                suspects.append(n)
                continue

            # 3. Check for Outlier Relationships (Adversarial Rewire)
            # If a restaurant is connected to something that isn't an Addr, City, or Phone
            valid_rels = {"city", "phone", "addr", "type", "name", "same_entity"}
            for _, _, d in out_edges:
                if d.get("relation") not in valid_rels:
                    suspects.append(n)
                    break
                    
        return list(set(suspects))
    
    def apply(self, g: nx.MultiDiGraph, ops: List[RepairOp]) -> Dict[str, int]:
        """Executes the code to actually modify the graph."""
        results = {}
        for op in ops:
            if op.kind == "schema_drift_repair":
                g = schema_drift_repair(
                    g, 
                    restaurant_id=op.params.get("node_id"), 
                    to_label=op.params.get("to_label")
                )
                
            elif op.kind == "entity_split_repair":
                g = entity_split_repair(
                    g, 
                    keep_node=op.params.get("keep_node"), 
                    drop_node=op.params.get("drop_node")
                )
                
            elif op.kind == "neighbor_rewire_repair":
                g = neighbor_rewire_repair(
                    g,
                    source_node=op.params.get("source_node"),
                    wrong_target=op.params.get("wrong_target"),
                    correct_target=op.params.get("correct_target"),
                    edge_key=op.params.get("edge_key", 0)
                )
                results[op.kind] = results.get(op.kind, 0) + 1
            elif op.kind == "entity_merge_repair":
                g = entity_merge_repair(
                    g,
                    node_to_split=op.params.get("node_to_split"),
                    new_node_id=op.params.get("new_node_id"),
                    move_neighbor_ids=op.params.get("move_neighbor_ids", [])
                )

            # 5. Pipeline ID Collision Repair
            elif op.kind == "pipeline_id_collision_repair":
                g = pipeline_id_collision_repair(
                    g,
                    node_id=op.params.get("node_id"),
                    new_unique_id=op.params.get("new_unique_id"),
                    attribute_to_split_by=op.params.get("attribute_to_split_by")
                )
            results[op.kind] = results.get(op.kind, 0) + 1
                
        return results

In [ ]:
repair_agent = RepairAgent()
g = g_corrupted.copy()
log_path = "corrupted_graphs/logs/fz_corruption_log_20260127-111352.json"
with open(log_path, 'r') as f:
    log_data = json.load(f)
findings = repair_agent.detect_hybrid_graph(g, corrupted_log=log_data, batch_size=5, sample_pct=0.01)
# print(f"Agent 1 (Detector) found {len(findings)} suspicious areas.")

# Step B: Planning
repair_ops = repair_agent.plan(g, findings)
# print(f"Agent 2 (Planner) proposed {len(repair_ops)} repair operations.")

# Step C: Application
repair_summary = repair_agent.apply(g, repair_ops)
print(f"Final Execution Results: {repair_summary}")

# 5. EVALUATION: Compare with Ground Truth
print(f"\n--- FINAL EVALUATION ---")

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("repaired_graphs")
OUTPUT_DIR.mkdir(exist_ok=True)
LOG_DIR = OUTPUT_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)

def save_graph(g: nx.MultiDiGraph, filename: str) -> None:
    data = {
        "nodes": [{"id": n, **g.nodes[n]} for n in g.nodes()],
        "edges": [{"src": u, "dst": v, **d} for u, v, d in g.edges(data=True)],
    }
    
    complete_path = OUTPUT_DIR / filename

    with open(complete_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def save_repair_log(findings, repair_ops, g, filename: str) -> None:
    data = {
        "findings": findings,
        "repair_ops": [str(o) for o in repair_ops],
        "final_graph_nodes": list(g.nodes(data=True))
    }

    complete_path = complete_path = LOG_DIR / filename

    with open(complete_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

In [ ]:
save_graph(g, f"fz_repaired_graph_full_{time.strftime('%Y%m%d-%H%M%S')}.json")
save_repair_log(findings, repair_ops, g, f"repair_log_full_{time.strftime('%Y%m%d-%H%M%S')}.json")